# 04 — Reduced Model (Top-10 SHAP Features)

Train a smaller CatBoost model restricted to the top 10 features by mean |SHAP|, to be used for prediction on external cohorts where the full ~730-feature annotation pipeline may be impractical to reproduce.

**Inputs**
- `TOPMed_cleaned_v4.csv` (from Notebook 02)
- `shap_feature_importance_rankings.csv` (from Notebook 03)

**Outputs**
- `models/reduced_top10/catboost_reduced_top10.cbm` / `.pkl` — final model (all data)
- `models/reduced_top10/cv_folds/` — 5 CV-fold models
- `results/reduced_top10/cv_predictions.csv`
- `results/reduced_top10/performance_summary.json`
- `results/reduced_top10/top10_features.json` — feature list + dtypes for inference


## 1. Imports & Configuration

In [1]:
import pandas as pd
import numpy as np
import json
import joblib
import yaml
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier, Pool
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    roc_auc_score, roc_curve, auc,
    precision_recall_curve, average_precision_score,
    confusion_matrix, precision_score, recall_score, f1_score,
)

In [4]:
config_path = Path("../config/config.yaml")
with open(config_path) as f:
    config = yaml.safe_load(f)

BASE_DIR = config_path.resolve().parent.parent

PATH_INPUT  = BASE_DIR / config['data']['cleaned']
TARGET      = config['model']['target']
RANDOM_SEED = config['model']['random_seed']
N_FOLDS     = config['model']['n_folds']
CATBOOST_PARAMS = config['model']['catboost'].copy()
CATEGORICAL_FEATURES_CONFIG = config['features']['categorical']

TOP_N = 10

SHAP_RANKINGS_PATH = BASE_DIR / config['shap']['rankings']

FIGURES_DIR = BASE_DIR / config['output']['figures_dir']
MODELS_DIR  = BASE_DIR / config['output']['models_dir']
CV_DIR      = MODELS_DIR / "cv_folds"
RESULTS_DIR = BASE_DIR / config['output']['results_dir']

for d in [MODELS_DIR, CV_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

Input data:       data/TOPMed_cleaned.csv
SHAP rankings:    results/figures/visualizations_cv/shap_manuscript/shap_feature_importance_rankings.csv
Models dir:       results/models/reduced_top10
Results dir:      results/reduced_top10
Target:           NMD.ESCAPEE
Top N features:   10
CV folds:         5
Random seed:      42


## 2. Load Cleaned Data & SHAP Rankings

In [6]:
# Cleaned data
df = pd.read_csv(PATH_INPUT)
y = df[TARGET].astype(int)
X_full = df.drop(columns=[TARGET])

print(f"✓ Loaded cleaned data: {df.shape}")
print(f"  Samples: {len(y)}")
print(f"  Escapees: {y.sum()} ({y.mean()*100:.1f}%)")
print(f"  Features available: {X_full.shape[1]}")

# SHAP rankings (already sorted by mean_abs_shap descending in Notebook 03)
shap_rank_df = pd.read_csv(SHAP_RANKINGS_PATH)
shap_rank_df = shap_rank_df.sort_values('mean_abs_shap', ascending=False).reset_index(drop=True)

print(f"\n✓ Loaded SHAP rankings: {len(shap_rank_df)} features")
print(f"\nTop {TOP_N} features by mean |SHAP|:")
print(shap_rank_df.head(TOP_N).to_string(index=False))

✓ Loaded cleaned data: (5749, 854)
  Samples: 5749
  Escapees: 2618 (45.5%)
  Features available: 853

✓ Loaded SHAP rankings: 853 features

Top 10 features by mean |SHAP|:
                          feature  mean_abs_shap
                         last.EJC       0.495423
              relativePTClocation       0.189621
                    half_life_PC1       0.150078
               cdsseqs_AU_content       0.119879
                         mut.exon       0.064799
phastcons_new3utr_first200_median       0.062054
         phylop_ptc_to_ejc_median       0.056727
                 AmountExonsAfter       0.056639
          cdsseq_AUcontentlast200       0.049026
               cdsseqs_UC_content       0.048924


## 3. Select Top-10 Features

In [7]:
# Pick top N features that actually exist in X_full (defensive)
top_features = shap_rank_df.head(TOP_N)['feature'].tolist()
missing = [f for f in top_features if f not in X_full.columns]
if missing:
    raise ValueError(f"Top features missing from cleaned data: {missing}")

X = X_full[top_features].copy()

# Identify which of these are categorical (per config list)
cat_features_selected = [f for f in top_features if f in CATEGORICAL_FEATURES_CONFIG]
cat_indices = [X.columns.get_loc(f) for f in cat_features_selected]

# CatBoost requires categorical columns as string (no NaN mixed with numerics)
for c in cat_features_selected:
    X[c] = X[c].astype(str).fillna("NA")

print(f"Reduced feature matrix: {X.shape}")
print(f"Categorical among top-{TOP_N}: {len(cat_features_selected)}")
for c in cat_features_selected:
    print(f"  - {c}")

Reduced feature matrix: (5749, 10)
Categorical among top-10: 1
  - last.EJC


## 4. Stratified 5-Fold Cross-Validation

In [8]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_SEED)

oof_preds = np.zeros(len(y))
fold_aucs = []
cv_models = []

params = CATBOOST_PARAMS.copy()
params['random_seed'] = RANDOM_SEED
params['verbose'] = False

print("="*80)
print(f"TRAINING REDUCED MODEL (top-{TOP_N} features) — {N_FOLDS}-fold CV")
print("="*80)

for fold_idx, (tr_idx, va_idx) in enumerate(skf.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    train_pool = Pool(X_tr, y_tr, cat_features=cat_indices)
    valid_pool = Pool(X_va, y_va, cat_features=cat_indices)

    model = CatBoostClassifier(**params)
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    probs = model.predict_proba(valid_pool)[:, 1]
    oof_preds[va_idx] = probs
    fold_auc = roc_auc_score(y_va, probs)
    fold_aucs.append(fold_auc)
    cv_models.append(model)

    print(f"  Fold {fold_idx+1}/{N_FOLDS}: AUC = {fold_auc:.4f}")

oof_auc = roc_auc_score(y, oof_preds)
oof_pr_auc = average_precision_score(y, oof_preds)
mean_auc = float(np.mean(fold_aucs))
std_auc  = float(np.std(fold_aucs))

print("\n" + "="*80)
print("CV RESULTS")
print("="*80)
print(f"  Out-of-fold AUC:    {oof_auc:.4f}")
print(f"  Out-of-fold PR-AUC: {oof_pr_auc:.4f}")
print(f"  Mean fold AUC:      {mean_auc:.4f} ± {std_auc:.4f}")

TRAINING REDUCED MODEL (top-10 features) — 5-fold CV
  Fold 1/5: AUC = 0.7588
  Fold 2/5: AUC = 0.7755
  Fold 3/5: AUC = 0.8011
  Fold 4/5: AUC = 0.7898
  Fold 5/5: AUC = 0.7715

CV RESULTS
  Out-of-fold AUC:    0.7793
  Out-of-fold PR-AUC: 0.7275
  Mean fold AUC:      0.7793 ± 0.0147


## 5. Classification Metrics at Youden-Optimal Threshold

In [9]:
fpr, tpr, thr = roc_curve(y, oof_preds)
youden = tpr - fpr
optimal_idx = int(np.argmax(youden))
THRESHOLD = float(thr[optimal_idx])

oof_labels = (oof_preds > THRESHOLD).astype(int)
prec = precision_score(y, oof_labels)
rec  = recall_score(y, oof_labels)
f1   = f1_score(y, oof_labels)
cm   = confusion_matrix(y, oof_labels)

print(f"Youden-optimal threshold: {THRESHOLD:.4f}")
print(f"\nConfusion matrix (rows=true, cols=pred):")
print(f"              Pred NMD   Pred Escape")
print(f"  True NMD    {cm[0,0]:>8d}    {cm[0,1]:>8d}")
print(f"  True Escape {cm[1,0]:>8d}    {cm[1,1]:>8d}")
print(f"\nPrecision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1:        {f1:.4f}")

Youden-optimal threshold: 0.4218

Confusion matrix (rows=true, cols=pred):
              Pred NMD   Pred Escape
  True NMD        2254         877
  True Escape      747        1871

Precision: 0.6809
Recall:    0.7147
F1:        0.6974


## 6. Compare to Full-Feature Model

In [10]:
# Pull baseline OOF AUC from Notebook 03 summary if available
full_summary_path = Path(config['output']['results_dir']) / "model_performance_summary.txt"
full_oof_auc = None
if full_summary_path.exists():
    txt = full_summary_path.read_text()
    for line in txt.splitlines():
        if "Out-of-Fold AUC" in line:
            try:
                full_oof_auc = float(line.split(":")[-1].strip())
            except Exception:
                pass
            break

print(f"Reduced (top-{TOP_N}) OOF AUC: {oof_auc:.4f}")
if full_oof_auc is not None:
    delta = oof_auc - full_oof_auc
    pct_retained = oof_auc / full_oof_auc * 100
    print(f"Full model OOF AUC:          {full_oof_auc:.4f}")
    print(f"Δ AUC:                       {delta:+.4f}")
    print(f"Performance retained:         {pct_retained:.1f}%")
else:
    print("(Full-model summary not found — run Notebook 03 first to populate baseline.)")

Reduced (top-10) OOF AUC: 0.7793
Full model OOF AUC:          0.7853
Δ AUC:                       -0.0060
Performance retained:         99.2%


## 7. Train Final Model on All Data & Save

In [11]:
print("Training final reduced model on all samples...")
final_pool = Pool(X, y, cat_features=cat_indices)
final_model = CatBoostClassifier(**params)
final_model.fit(final_pool)

# Save final model
final_cbm = MODELS_DIR / f"catboost_reduced_top{TOP_N}.cbm"
final_pkl = MODELS_DIR / f"catboost_reduced_top{TOP_N}.pkl"
final_model.save_model(str(final_cbm))
joblib.dump(final_model, final_pkl)
print(f"✓ Final model saved:")
print(f"  {final_cbm}")
print(f"  {final_pkl}")

# Save CV fold models
for i, m in enumerate(cv_models):
    fold_auc = fold_aucs[i]
    m.save_model(str(CV_DIR / f"fold_{i+1}_auc_{fold_auc:.4f}.cbm"))
    joblib.dump(m, CV_DIR / f"fold_{i+1}_auc_{fold_auc:.4f}.pkl")
print(f"✓ {N_FOLDS} CV fold models saved to: {CV_DIR}")

Training final reduced model on all samples...
✓ Final model saved:
  results/models/reduced_top10/catboost_reduced_top10.cbm
  results/models/reduced_top10/catboost_reduced_top10.pkl
✓ 5 CV fold models saved to: results/models/reduced_top10/cv_folds


## 8. Save Feature Metadata (for External Inference)

In [12]:
# This is the contract external cohorts must satisfy to use the model.
feature_metadata = {
    "model_name": f"spooky_model_reduced_top{TOP_N}",
    "target": TARGET,
    "n_features": TOP_N,
    "features_in_order": top_features,
    "categorical_features": cat_features_selected,
    "numerical_features": [f for f in top_features if f not in cat_features_selected],
    "shap_ranking_source": str(SHAP_RANKINGS_PATH),
    "mean_abs_shap": {
        row['feature']: float(row['mean_abs_shap'])
        for _, row in shap_rank_df.head(TOP_N).iterrows()
    },
    "youden_threshold": THRESHOLD,
    "training_samples": int(len(y)),
    "escape_rate": float(y.mean()),
    "random_seed": RANDOM_SEED,
    "catboost_params": {k: v for k, v in params.items() if k != "verbose"},
}

feat_path = RESULTS_DIR / f"top{TOP_N}_features.json"
with open(feat_path, "w") as f:
    json.dump(feature_metadata, f, indent=2, default=str)
print(f"✓ Feature metadata saved: {feat_path}")

✓ Feature metadata saved: results/reduced_top10/top10_features.json


## 9. Save Predictions & Performance Summary

In [13]:
# CV predictions
cv_pred_df = pd.DataFrame({
    "index": np.arange(len(y)),
    "y_true": y.values,
    "oof_prob": oof_preds,
    "oof_pred_at_youden": oof_labels,
})
cv_pred_path = RESULTS_DIR / "cv_predictions.csv"
cv_pred_df.to_csv(cv_pred_path, index=False)
print(f"✓ CV predictions: {cv_pred_path}")

# Performance summary
summary = {
    "model": f"reduced_top{TOP_N}",
    "n_features": TOP_N,
    "features": top_features,
    "n_samples": int(len(y)),
    "escape_rate": float(y.mean()),
    "cv_oof_auc": float(oof_auc),
    "cv_oof_pr_auc": float(oof_pr_auc),
    "cv_mean_fold_auc": mean_auc,
    "cv_std_fold_auc": std_auc,
    "cv_fold_aucs": [float(a) for a in fold_aucs],
    "youden_threshold": THRESHOLD,
    "precision_at_threshold": float(prec),
    "recall_at_threshold": float(rec),
    "f1_at_threshold": float(f1),
    "confusion_matrix": cm.tolist(),
    "full_model_oof_auc": float(full_oof_auc) if full_oof_auc is not None else None,
    "delta_vs_full": float(oof_auc - full_oof_auc) if full_oof_auc is not None else None,
}
summary_path = RESULTS_DIR / "performance_summary.json"
with open(summary_path, "w") as f:
    json.dump(summary, f, indent=2)
print(f"✓ Performance summary: {summary_path}")

✓ CV predictions: results/reduced_top10/cv_predictions.csv
✓ Performance summary: results/reduced_top10/performance_summary.json


## 10. Inference Helper for External Cohort Variants

Minimal reference snippet — copy into a prediction script. Requires the external cohort dataframe to have all 10 feature columns; categoricals should be cast to string and missing values encoded as `"NA"` to match training.

```python
import joblib, json, pandas as pd
from catboost import Pool

meta = json.load(open("results/reduced_top10/top10_features.json"))
model = joblib.load("models/reduced_top10/catboost_reduced_top10.pkl")

cohort_df = pd.read_csv("path/to/external_cohort.csv")
X_new = cohort_df[meta["features_in_order"]].copy()
for c in meta["categorical_features"]:
    X_new[c] = X_new[c].astype(str).fillna("NA")

cat_idx = [X_new.columns.get_loc(c) for c in meta["categorical_features"]]
probs = model.predict_proba(Pool(X_new, cat_features=cat_idx))[:, 1]
cohort_df["escape_prob"] = probs
cohort_df["escape_pred"] = (probs > meta["youden_threshold"]).astype(int)
```


In [14]:
print("="*80)
print("DONE")
print("="*80)
print(f"Reduced model (top-{TOP_N}): OOF AUC = {oof_auc:.4f}")
if full_oof_auc is not None:
    print(f"vs full model OOF AUC = {full_oof_auc:.4f} (Δ = {oof_auc - full_oof_auc:+.4f})")
print(f"\nAll outputs under:")
print(f"  {MODELS_DIR}")
print(f"  {RESULTS_DIR}")

DONE
Reduced model (top-10): OOF AUC = 0.7793
vs full model OOF AUC = 0.7853 (Δ = -0.0060)

All outputs under:
  results/models/reduced_top10
  results/reduced_top10
